# `config.ipynb` — Configuración del proyecto GeoStat

Módulo de configuración: todas las credenciales y parámetros del proyecto en un único fichero, para que sea fácil localizarlos y modificarlos.

Los valores por defecto coinciden exactamente con `docker-compose.yml` (puerto **5433**, requisito de infraestructura) y con la ubicación de `main.ipynb` en la raíz del proyecto (al mismo nivel que la carpeta `geostat_etl/`), pero todos admiten sobreescritura por variable de entorno - así el mismo módulo sirve tanto para el despliegue final en Docker como para pruebas locales, sin tocar el código.

> Este notebook se ejecuta con `%run geostat_etl/config.ipynb` desde `main.ipynb`, que es el mecanismo que usa este proyecto para encadenar los módulos (los `.ipynb` no se pueden `import`ar como un `.py`). Todas las constantes que defina esta celda quedan disponibles para el resto de módulos y para `main.ipynb`.

In [ ]:
import os

## 1. Conexión a PostgreSQL (destino)

In [ ]:
DB_HOST = os.environ.get("GEOSTAT_DB_HOST", "localhost")
DB_PORT = os.environ.get("GEOSTAT_DB_PORT", "5433")
DB_NAME = os.environ.get("GEOSTAT_DB_NAME", "db_geostat_dw")
DB_USER = os.environ.get("GEOSTAT_DB_USER", "geostat_user")
DB_PASSWORD = os.environ.get("GEOSTAT_DB_PASSWORD", "geostat_pass")

print(f"Destino PostgreSQL configurado -> host={DB_HOST}, puerto={DB_PORT}, bd={DB_NAME}, usuario={DB_USER}")

## 2. Origen de datos Legacy (SQLite)

Ruta relativa a la raíz del proyecto (donde vive `main.ipynb`). `CHUNK_SIZE` es el tamaño de lote con el que se leerá la tabla `economia_paises` (20.000 filas) - requisito de **procesamiento eficiente por chunks**.

In [ ]:
SQLITE_PATH = os.environ.get("GEOSTAT_SQLITE_PATH", "./datos_economicos_locales.db")
SQLITE_TABLA_ORIGEN = "economia_paises"

CHUNK_SIZE = int(os.environ.get("GEOSTAT_CHUNK_SIZE", "2000"))

print(f"Origen SQLite -> {SQLITE_PATH} (tabla: {SQLITE_TABLA_ORIGEN}), chunk_size={CHUNK_SIZE}")

## 3. API REST de datos demográficos

Endpoint indicado en la especificación. `API_MAX_REINTENTOS` controla cuántas veces se reintenta antes de activar el mecanismo de respaldo (fallback) - ver `demografia.ipynb`.

In [ ]:
API_DEMOGRAFIA_URL = "https://restcountries.com/v3.1/region/europe"
API_TIMEOUT_SEGUNDOS = 8
API_MAX_REINTENTOS = 2

## 4. Logging

Ruta del fichero de log. El logger en sí se configura en `logging_config.ipynb`.

In [ ]:
RUTA_LOG = os.environ.get("GEOSTAT_RUTA_LOG", "./logs/ejecucion_etl.log")

## 5. Reglas de transformación - unidades y divisas

- **Superficie**: la fuente Legacy mezcla `sq_mi` (millas cuadradas) y `km2`. Factor de conversión estándar: **1 sq mi = 2.58999 km²**.
- **Divisas**: la fuente Legacy usa 11 divisas distintas. Se define una **estructura fija de tipos de cambio frente al EUR** (referencia BCE, formato `1 EUR = X divisa`).

In [ ]:
FACTOR_SQMI_A_KM2 = 2.58999

# Tipos de cambio frente al EUR (1 EUR = X unidades de la divisa)
TASAS_CAMBIO_EUR = {
    "EUR": 1.0,
    "USD": 1.1614,
    "GBP": 0.85740,
    "CHF": 0.9425,
    "SEK": 11.1520,
    "NOK": 10.7450,
    "DKK": 7.4748,
    "CZK": 24.186,
    "PLN": 4.3178,
    "HUF": 363.95,
    "BGN": 1.95583,   # BGN mantiene una paridad fija histórica con el EUR
}

## 6. Normalización de nombres de país - sinónimos

La fuente Legacy mezcla nombres en inglés y en español (`España`, `Francia`). Este diccionario traduce las variantes en español a su forma canónica en inglés, después de haber quitado acentos y pasado a mayúsculas (ver `transform.ipynb`).

In [ ]:
SINONIMOS_PAIS = {
    "ESPANA": "SPAIN",
    "FRANCIA": "FRANCE",
    "ALEMANIA": "GERMANY",
    "REINO UNIDO": "UNITED KINGDOM",
    "PAISES BAJOS": "NETHERLANDS",
    "SUIZA": "SWITZERLAND",
    "SUECIA": "SWEDEN",
    "DINAMARCA": "DENMARK",
    "GRECIA": "GREECE",
    "POLONIA": "POLAND",
    "HUNGRIA": "HUNGARY",
    "REPUBLICA CHECA": "CZECHIA",
    "IRLANDA": "IRELAND",
    "ITALIA": "ITALY",
    "BELGICA": "BELGIUM",
    "NORUEGA": "NORWAY",
    "FINLANDIA": "FINLAND",
    "PORTUGAL": "PORTUGAL",
    "RUMANIA": "ROMANIA",
    "BULGARIA": "BULGARIA",
    "CROACIA": "CROATIA",
}

## 7. Dataset de respaldo (fallback) de población

Se activa **únicamente** si la API REST no está disponible o falla la conexión HTTP (requisito de resiliencia - ver `demografia.ipynb`). Contiene la población total de los 44 países europeos reconocidos por el proyecto.

> **Fuente**: estimaciones de Naciones Unidas - *World Population Prospects 2025* (Wikipedia, *List of European countries by population*), salvo Ciudad del Vaticano (cifra oficial de la Santa Sede, sin estimación ONU disponible).

In [ ]:
POBLACION_FALLBACK = {
    "ALBANIA": 2_771_508,
    "ANDORRA": 82_904,
    "AUSTRIA": 9_113_574,
    "BELARUS": 8_997_603,
    "BELGIUM": 11_758_603,
    "BOSNIA AND HERZEGOVINA": 3_140_096,
    "BULGARIA": 6_714_560,
    "CROATIA": 3_848_160,
    "CYPRUS": 1_370_754,
    "CZECHIA": 10_609_240,
    "DENMARK": 6_002_507,
    "SPAIN": 47_889_958,
    "ESTONIA": 1_344_232,
    "FINLAND": 5_623_330,
    "FRANCE": 66_650_804,
    "GERMANY": 84_075_074,
    "GREECE": 9_938_844,
    "HUNGARY": 9_632_287,
    "ICELAND": 398_266,
    "IRELAND": 5_308_039,
    "ITALY": 59_146_260,
    "LATVIA": 1_853_559,
    "LIECHTENSTEIN": 40_128,
    "LITHUANIA": 2_830_144,
    "LUXEMBOURG": 680_454,
    "MALTA": 545_405,
    "MOLDOVA": 2_996_106,
    "MONACO": 38_341,
    "MONTENEGRO": 632_729,
    "NETHERLANDS": 18_346_819,
    "NORTH MACEDONIA": 1_813_791,
    "NORWAY": 5_618_354,
    "POLAND": 37_332_000,
    "PORTUGAL": 10_411_834,
    "ROMANIA": 18_908_650,
    "SAN MARINO": 33_572,
    "SERBIA": 6_689_039,
    "SLOVAKIA": 5_474_881,
    "SLOVENIA": 2_117_072,
    "SWEDEN": 10_656_633,
    "SWITZERLAND": 8_967_408,
    "UKRAINE": 38_980_377,
    "UNITED KINGDOM": 69_551_332,
    "VATICAN CITY": 764,
}

# Conjunto de países europeos válidos (claves normalizadas). Cualquier
# entidad de la fuente Legacy que no aparezca aquí tras normalizar y
# traducir se considera "no perteneciente a la región europea" y va a
# cuarentena (p. ej. entidades ficticias como Narnia o Atlantis).
PAISES_EUROPA_VALIDOS = set(POBLACION_FALLBACK.keys())

print(f"Dataset de respaldo cargado: {len(POBLACION_FALLBACK)} paises europeos")